## Loads all the Chlorophyll Satellite data and saves it as a csv
Last edited by L. Gruenburg on  June 4, 2024

In [1]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import xarray as xr
import cartopy
import pandas as pn
from datetime import datetime
import geopandas as gpd
from shapely.geometry import Polygon, Point, MultiPolygon
import shapefile
import cartopy
import cartopy.crs as ccrs
import numpy as np
import copernicusmarine
import regionmask
#import holoviews as hv
#from holoviews import opts
#from xmovie import Movie
#hv.extension('bokeh', 'matplotlib')
#hv.extension('matplotlib')

The surface chlorophyll data are from the TERRA/MODIS satellite mission details here https://oceancolor.gsfc.nasa.gov/about/missions/terra
You can download any updates to the data manually from this website

In [ ]:
chla = copernicusmarine.open_dataset(
  dataset_id="cmems_obs-oc_glo_bgc-plankton_my_l4-multi-4km_P1M",
  variables=["CHL_uncertainty", "CHL", "flags"],
  minimum_longitude=-75,
  maximum_longitude=-68,
  minimum_latitude=38,
  maximum_latitude=42.5,
  start_datetime="1997-01-09T00:00:00",
  end_datetime="2025-01-11T23:59:00",
)

In [ ]:
#chla.to_netcdf('chla_2025.nc')

In [2]:
chla = xr.open_dataset('chla_2025.nc')

In [3]:
# Load NYB shapefile
NYB = gpd.read_file('~/Desktop/NYB_Indicators_Calculations/Datasets/Shapefiles/PlanningArea_NYocean_NYSDOS.shp')
# Alter the projection to WGS84 see https://epsg.io/4326
NYB = NYB.to_crs(epsg=4326)
# Extract the polygon
nyb_shape = NYB.geometry[0]

In [4]:
# function for cropping the data to the shape
def crop_nd(data, longitude_name, latitude_name, shape):
    
    # Get the region of interest
    region = regionmask.from_geopandas(shape)
    
    # Create the mask
    mask = region.mask(data[longitude_name].astype('f4'), data[latitude_name].astype('f4'))
    
    # Apply mask to the data
    masked_ds = data.where(mask == region.numbers[0])
    
    return masked_ds 

In [5]:
chla_nyb = crop_nd(chla, 'longitude', 'latitude', NYB)

In [6]:
chla_nyb_mean = chla_nyb.mean(['latitude','longitude'])

In [12]:
#year_pickedNYB = chla_nyb_mean.CHL[chla_nyb_mean.time.dt.year == i]
#unique_months = np.unique(year_pickedNYB.time.dt.month)
unique_years = np.unique(chla_nyb_mean.time.dt.year)
unique_years

array([1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007,
       2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018,
       2019, 2020, 2021, 2022, 2023, 2024, 2025])

In [13]:
chla_nyb_mean

<xarray.Dataset> Size: 8kB
Dimensions:          (time: 329)
Coordinates:
  * time             (time) datetime64[ns] 3kB 1997-09-01 ... 2025-01-01
Data variables:
    CHL_uncertainty  (time) float64 3kB 15.65 8.705 12.2 ... 5.329 6.225 5.71
    CHL              (time) float32 1kB 0.6162 1.033 1.24 ... 1.096 1.085 1.053
    flags            (time) float32 1kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0

In [18]:
np.append(np.nan,chla_nyb_mean.CHL[chla_nyb_mean.time.dt.month == 1][:27])

array([       nan, 1.09554255, 1.39289129, 0.99439973, 1.5105617 ,
       1.22880697, 1.08281839, 1.43236864, 1.17674649, 2.38293266,
       1.3360827 , 1.94499123, 1.0941993 , 1.20125461, 1.36306369,
       1.24255514, 1.0728296 , 1.09152424, 1.07423043, 1.20282316,
       1.07688189, 1.40177929, 1.44013977, 1.17784095, 1.2604183 ,
       1.10197222, 1.04688096, 1.7196269 ])

In [30]:
chla_monthly = pn.DataFrame({'Jan': np.append(np.nan,chla_nyb_mean.CHL[chla_nyb_mean.time.dt.month == 1]),
              'Feb': np.append(np.nan,np.append(chla_nyb_mean.CHL[chla_nyb_mean.time.dt.month == 2],np.nan)),
              'Mar': np.append(np.nan,np.append(chla_nyb_mean.CHL[chla_nyb_mean.time.dt.month == 3],np.nan)),
              'Apr': np.append(np.nan,np.append(chla_nyb_mean.CHL[chla_nyb_mean.time.dt.month == 4],np.nan)),
              'May': np.append(np.nan,np.append(chla_nyb_mean.CHL[chla_nyb_mean.time.dt.month == 5],np.nan)),
              'Jun': np.append(np.nan,np.append(chla_nyb_mean.CHL[chla_nyb_mean.time.dt.month == 6],np.nan)),
              'Jul': np.append(np.nan,np.append(chla_nyb_mean.CHL[chla_nyb_mean.time.dt.month == 7],np.nan)),
              'Aug': np.append(np.nan,np.append(chla_nyb_mean.CHL[chla_nyb_mean.time.dt.month == 8],np.nan)),
              'Sep': np.append(chla_nyb_mean.CHL[chla_nyb_mean.time.dt.month == 9],np.nan),
              'Oct': np.append(chla_nyb_mean.CHL[chla_nyb_mean.time.dt.month == 10],np.nan),
              'Nov': np.append(chla_nyb_mean.CHL[chla_nyb_mean.time.dt.month == 11],np.nan),
              'Dec': np.append(chla_nyb_mean.CHL[chla_nyb_mean.time.dt.month == 12],np.nan)},
             index = np.unique(chla_nyb_mean.time.dt.year))


In [31]:
chla_monthly

,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
1997,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.616197,1.032888,1.240269,1.015888
1998,1.095543,0.980160,1.456217,1.503487,0.993013,0.764018,0.716068,0.503458,0.596115,1.048392,2.056230,1.688787
1999,1.392891,1.237292,1.703279,1.488346,1.029427,0.661940,0.700643,0.919194,0.825476,1.239492,1.792241,1.374093
2000,0.994400,0.932798,1.036935,1.935207,1.619478,1.210372,1.498363,0.975670,0.800109,1.102681,1.473310,1.689753
2001,1.510562,1.503320,1.313551,2.291547,1.316978,1.324656,1.047571,0.726260,0.599362,1.057022,1.181958,1.202525
2002,1.228807,1.212502,1.376758,1.470507,1.291762,0.934798,1.798625,1.420690,0.843278,0.863085,1.047241,0.877887
2003,1.082818,0.948643,1.542059,1.686288,1.314680,0.834848,0.677809,1.838283,0.920693,1.041282,1.456999,1.251552
2004,1.432369,1.086549,1.189331,1.224106,1.079002,0.719119,0.660337,0.734092,0.758513,1.278882,1.955253,1.210583
2005,1.176746,1.008787,1.023917,1.244982,0.890954,0.694199,0.670996,0.608767,0.769212,1.224507,2.636363,1.731837
2006,2.382933,1.425956,1.932183,1.594463,0.830164,0.934261,0.882581,1.220996,0.819124,1.104536,1.565835,1.278162


In [32]:
chla_monthly.to_csv('chla_monthly_2025.csv')

In [28]:
jan2025 =pn.DataFrame({'Jan': chla_nyb_mean.CHL[chla_nyb_mean.time.dt.month == 1][27],
              'Feb': np.nan,
              'Mar': np.nan,
              'Apr': np.nan,
              'May': np.nan,
              'Jun': np.nan,
              'Jul': np.nan,
              'Aug': np.nan,
              'Sep': np.nan,
              'Oct': np.nan,
              'Nov': np.nan,
              'Dec': np.nan},
             index = 2025)

TypeError: Index(...) must be called with a collection of some kind, 2025 was passed

In [ ]:
unique_years = np.unique(chla_nyb_mean.time.dt.year)
monthly_nyb = np.empty([len(unique_years),12])
k=0

for i in (unique_years):
    year_pickedNYB = chla_nyb_mean.CHL[chla_nyb_mean.time.dt.year == i]
    unique_months = np.unique(year_pickedNYB.time.dt.month)
    for j in range(len(unique_months)):
        monthly_nyb[k,unique_months[j]-1] = year_pickedNYB[j]
    k = k+1
len(monthly_nyb)
len(unique_years)

In [ ]:
#a little test just to see what the data looks like
chlatest = xr.open_dataset('/Users/nyelab/Downloads/requested_files 2/TERRA_MODIS.20230801_20230831.L3m.MO.CHL.x_chlor_a.nc')

In [ ]:
chlatest

In [ ]:
#These files live on the NYOS megafolder in the google drive in NYOS_megafolder/indicator.development/Indicators_MOST_RECENT_2023/Raw Data/Chlorophyll
import os
files = os.listdir('/Users/lauragruenburg/Downloads/CHLA_2025/')
len(files)

In [ ]:
files

In [ ]:
xr.open_dataset('~/Downloads/CHLA_2025/TERRA_MODIS.20070201_20070228.L3b.MO.CHL.x.nc')

In [ ]:
# Merge the data from all teh separate files together
chla_data = np.empty([120,312, 309])
chla_data[:,:,:] = np.nan
chla_month = np.empty(309)
chla_month[:] = np.nan
chla_year = np.empty(309)
chla_year[:] = np.nan
chla_date = 'start'

for i in range(283):
    file_name = files[i]
    chl = xr.open_dataset('/Users/nyelab/Downloads/requested_files 2/' + file_name)
    date = pn.to_datetime((getattr(chl,'time_coverage_end')))
    month = pn.to_datetime((getattr(chl,'time_coverage_end'))).month
    year = pn.to_datetime((getattr(chl,'time_coverage_end'))).year
    chla_data[:,:,i] = chl.chlor_a
    chla_month[i] = month
    chla_year[i] = year
    chla_date = np.append(chla_date, str(month) + '-' + str(year))

In [ ]:
#Double checking the date variable
chla_date = chla_date[1:]
chla_t = pn.to_datetime(chla_date)
len(chla_date)

In [ ]:

#i=0
#m = np.sort(pn.to_datetime(chla_date))[i]
#m
#index = np.where(chla_t == m)
#index[0][0]

In [ ]:
# The data is not in chronologicl order, this step reorders the data
chla_in_order = np.empty([120,312, 283])
chla_in_order[:,:,:] = np.nan

for i in range(283):
    m = np.sort(pn.to_datetime(chla_date))[i]
    index = np.where(chla_t == m)
    chla_in_order[:,:,i] = chla_data[:,:,index[0][0]]
    

In [ ]:
# Create a dataset for the chlorohpyll data
CH = xr.Dataset(data_vars ={'chla': (['lat','lon','time'], chla_in_order)},
               coords = {'lat': chl.lat,
                        'lon': chl.lon,
                        'time': np.sort(pn.to_datetime(chla_date))})

In [ ]:
# make a plot to look at it
plt.plot(CH.time, np.nanmean(np.nanmean(CH.chla, 0),0))

In [ ]:
# Load NYB shapefile
NYB = gpd.read_file('/Users/nyelab/Desktop/NYB Indicators/Datasets/Shapefiles/PlanningArea_NYocean_NYSDOS.shp')
# Alter the projection to WGS84 see https://epsg.io/4326
NYB = NYB.to_crs(epsg=4326)
# Extract the polygon
nyb_shape = NYB.geometry[0]

In [ ]:
#empty matrix for the function below
NYB_new = np.empty([120,312,283])
NYB_new[:,:,:]=np.nan


In [ ]:
#function to only extract data in our EPU
def inEPU(empty_matrix, shape, lon_mat_name, lat_mat_name,variable_matrix):
    for i in range(120):
        for j in range(312):
            if Point(lon_mat_name[j], lat_mat_name[i]).within(shape) == True:
                empty_matrix[i,j,:] = variable_matrix[i,j,:]
    return empty_matrix

In [ ]:
#keeping only surface chla data for the nyb
nyb_chla = inEPU(NYB_new, nyb_shape, CH.lon, CH.lat, CH.chla)

In [ ]:
# spatially averaged timeseries of the nyb surfcae chlorophyll
chla_ts = np.nanmean(np.nanmean(nyb_chla,0),0)

In [ ]:
plt.plot(CH.time, chla_ts)

In [ ]:
#Create the dataframe of this nyb spatially averaged ts for export to .csv
chlorophyll = pn.DataFrame({'date': CH.time,
             'chla': chla_ts})

In [ ]:
#Export to .csv for final plotting
chlorophyll.to_csv('/Users/nyelab/Desktop/NYB Indicators/NYB_Indicators_Calculations/Final_Timeseries_Figures/Timeseries_Files_2023/chla_2023.csv')

## Everything below this is not necessary for the indicators

In [ ]:
%matplotlib inline
plt.imshow(chlatest.chlor_a, vmin = 0, vmax = 10)
plt.colorbar()

In [ ]:
#chla = xr.open_dataset('/Users/nyelab/Downloads/CCI_ALL-v5.0-MONTHLY.nc')
chla = xr.open_dataset('/Users/nyelab/Downloads/pmlEsaCCI50OceanColorMonthly_82cc_542d_f222.nc')

In [ ]:
chla

In [ ]:
# Load NYB shapefile
NYB = gpd.read_file('/Users/nyelab/Desktop/NYB Indicators/Datasets/Shapefiles/PlanningArea_NYocean_NYSDOS.shp')
# Alter the projection to WGS84 see https://epsg.io/4326
NYB = NYB.to_crs(epsg=4326)
# Extract the polygon
nyb_shape = NYB.geometry[0]

In [ ]:
# Load MAB and GoM shapefiles (already in WGS84).
# Note that this also contains shapefiles for the Scotia Shelf at index 2 and the Grand Banks at index 3.
EPU = gpd.read_file('/Users/nyelab/Desktop/NYB Indicators/Datasets/Shapefiles/EPU_NOESTUARIES.shp')
mab_shape = EPU.geometry[0]
gom_shape = EPU.geometry[1]

In [ ]:
NYB

In [ ]:
(lon, lat) = np.meshgrid(chla.longitude,chla.latitude)

In [ ]:
NYB = np.empty([409,481])
NYB[:,:]=np.nan
EPU = np.empty([409,481])
EPU[:,:]=np.nan

In [ ]:
def inEPU(empty_matrix, shape, lon_mat_name, lat_mat_name,value):
    for i in range(409):
        for j in range(481):
            if Point(lon_mat_name[i,j], lat_mat_name[i,j]).within(shape) == True:
                empty_matrix[i,j] = value
    return empty_matrix

In [ ]:
NYB = inEPU(NYB,nyb_shape, lon,lat,1)
#EPU = inEPU(EPU,mab_shape,lon,lat,1)
#EPU = inEPU(EPU,gom_shape,lon,lat,2)

In [ ]:
NYB = xr.DataArray(data = NYB, dims={'latitude': 409, 'longitude': 481}, coords = {'latitude':chla.latitude,'longitude': chla.longitude})
nyb = NYB.to_dataset(name='nyb')
#EPU = xr.DataArray(data = EPU, dims={'lat': 409, 'lon': 481}, coords = {'lat':chla.lat,'lon': chla.lon})
#epu = EPU.to_dataset(name='epu')

In [ ]:
chla = xr.merge([chla, nyb, epu])

In [ ]:
dates = pn.DatetimeIndex(chla.time.values)
unique_years = np.unique(dates.year)

In [ ]:
nyb

In [ ]:
chlaMAB = np.empty([280,409,481])
chlaGOM = np.empty([280,409,481])

for j in range(409):
    for k in range(481):
        if chla.epu[j,k] == 1:
            chlaMAB[:,j,k] = chla.chlor_a[:,j,k]
            chlaGOM[:,j,k] = np.nan
        if chla.epu[j,k] == 2:
            chlaMAB[:,j,k] = np.nan
            chlaGOM[:,j,k] = chla.chlor_a[:,j,k]
        if chla.epu[j,k] != 1 and chla.epu[j,k] != 2:
            chlaMAB[:,j,k] = np.nan
            chlaGOM[:,j,k] = np.nan

In [ ]:
chlaNYB = np.empty([292,409,481])
for j in range(409):
    for k in range(481):
        if nyb.nyb[j,k] == 1:
            chlaNYB[:,j,k] = chla.chlor_a[:,j,k]
        else:
            chlaNYB[:,j,k] = np.nan


In [ ]:
# Creates matricies of monthoy means the data by year
#monthly_gom = np.empty([len(unique_years),12])
#monthly_mab = np.empty([len(unique_years),12])
monthly_nyb = np.empty([len(unique_years),12])
k=0
for i in (unique_years):
    #year_pickedMAB = np.nanmean(np.nanmean(chlaMAB[dates.year == i,:,:],1),1)
    #year_pickedGOM = np.nanmean(np.nanmean(chlaGOM[dates.year == i,:,:],1),1)
    year_pickedNYB = np.nanmean(np.nanmean(chlaNYB[dates.year == i,:,:],1),1)
    months_picked = dates[dates.year == i].month
    for j in range(len(months_picked)):
        #monthly_gom[k,months_picked[j]-1] = year_pickedGOM[j]
        #monthly_mab[k,months_picked[j]-1] = year_pickedMAB[j]
        monthly_nyb[k,months_picked[j]-1] = year_pickedNYB[j]
    k = k+1

In [ ]:
len(monthly_nyb)
len(unique_years)
d = {'jan': monthly_nyb[:,0], 'feb': monthly_nyb[:,1], 'mar': monthly_nyb[:,2],
    'apr': monthly_nyb[:,3], 'may': monthly_nyb[:,4], 'jun': monthly_nyb[:,5],
    'jul': monthly_nyb[:,6], 'aug': monthly_nyb[:,7], 'sep': monthly_nyb[:,8],
    'oct': monthly_nyb[:,9], 'nov': monthly_nyb[:,10], 'dec': monthly_nyb[:,11]}
chla_nyb = pn.DataFrame(data = d, index = unique_years)
chla_nyb.to_csv('/Users/nyelab/Desktop/NYB Indicators/FInal_Timeseries/chla_nyb_12_14_2022.csv')


In [ ]:
chla_nyb

In [ ]:
chla_nyb.to_csv('/Users/nyelab/Desktop/NYB Indicators/FInal_Timeseries/chla_nyb_12_14_2022.csv')

In [ ]:
np.shape(monthly_nyb)

In [ ]:
fig, ax = plt.subplots(figsize = [10,10])

#ax.plot(unique_years, monthly_mab[:,1], 'purple', linestyle = (0,(3,5,1,5)))
ax.plot(unique_years[1:], monthly_mab[1:,2], 'purple', linestyle = ':', label = "MAB Mar")
ax.plot(unique_years[1:], monthly_mab[1:,3], 'purple', linestyle = '-', label = 'MAB Apr')
ax.plot(unique_years[1:], monthly_mab[1:,4], 'purple', linestyle = '-.', label = 'MAB May')

ax.set_yticks(np.arange(-1,2.5,0.5))
ax.set_yticklabels(['','','',0.5,1.0,1.5,2.0], fontsize = 14, color = 'purple')
ax.set_ylabel('Chla ($\mathregular{mg m^{-3}}$)', fontsize = 14, y=0.75)

ax2 = plt.twinx(ax)

ax2.plot(unique_years[1:], monthly_gom[1:,2], 'green', linestyle = ':', label = 'GoM Mar')
ax2.plot(unique_years[1:], monthly_gom[1:,3], 'green', linestyle = '-', label = 'GoM Apr')
ax2.plot(unique_years[1:], monthly_gom[1:,4], 'green', linestyle = '-.', label = 'GoM May')

ax2.set_ylim([0.5,4])
ax2.set_yticks([0.5, 1.0, 1.5, 2.0])
ax2.set_yticklabels([0.5, 1.0, 1.5, 2.0], fontsize = 14, color = 'green')

ax.legend(fontsize = 14, loc = 'upper left')
ax2.legend(fontsize = 14, loc = 'center right')
ax.set_xticks(np.arange(1998,2022,2))
ax.set_xticklabels(np.arange(1998,2022,2), fontsize = 14)
ax2.set_ylabel('Chla ($\mathregular{mg m^{-3}}$)', fontsize = 14, y=0.25)


In [ ]:
fig, ax = plt.subplots(figsize = [10,7])

#ax.plot(unique_years, monthly_mab[:,1], 'purple', linestyle = (0,(3,5,1,5)))
ax.plot(unique_years[1:], monthly_nyb[1:,2], 'orange', linestyle = ':', label = "March")
ax.plot(unique_years[1:], monthly_nyb[1:,3], 'orange', linestyle = '-', label = 'April')
ax.plot(unique_years[1:], monthly_nyb[1:,4], 'orange', linestyle = '-.', label = 'May')

ax.set_yticks(np.arange(0.6,2,0.2))
ax.tick_params(labelsize = 14)
#ax.set_yticklabels(['','','',0.5,1.0,1.5,2.0], fontsize = 14, color = 'purple')
ax.set_ylabel('Chla ($\mathregular{mg m^{-3}}$)', fontsize = 14, y=0.5)
ax.set_xticks(np.arange(1998,2022,2))
ax.set_xticklabels(np.arange(1998,2022,2), fontsize = 14)
ax.legend(fontsize = 14, loc = 'upper left')

In [ ]:
fig, ax = plt.subplots(figsize = [10,7])

#ax.plot(unique_years, monthly_mab[:,1], 'purple', linestyle = (0,(3,5,1,5)))
ax.plot(unique_years[1:], monthly_nyb[1:,2], 'k', linestyle = ':', label = "March")
ax.plot(unique_years[1:], monthly_nyb[1:,3], 'k', linestyle = '-', label = 'April')
ax.plot(unique_years[1:], monthly_nyb[1:,4], 'k', linestyle = '-.', label = 'May')

ax.set_yticks(np.arange(0.6,2,0.2))
ax.tick_params(labelsize = 14)
#ax.set_yticklabels(['','','',0.5,1.0,1.5,2.0], fontsize = 14, color = 'purple')
ax.set_ylabel('Chl-a ($\mathregular{mg m^{-3}}$)', fontsize = 14, y=0.5)
ax.set_xticks(np.arange(1998,2022,2))
ax.set_xticklabels(np.arange(1998,2022,2), fontsize = 14)
ax.legend(fontsize = 14, loc = 'upper left')

In [ ]:
# plot the actual years
mab_ts = monthly_mab[0,:]
nyb_ts = monthly_nyb[0,:]
gom_ts = monthly_gom[0,:]
for i in range(1,24):
    mab_ts = np.append(mab_ts,monthly_mab[i,:])
    nyb_ts = np.append(nyb_ts,monthly_nyb[i,:])
    gom_ts = np.append(gom_ts,monthly_gom[i,:])

In [ ]:
nyb_ts= nyb_ts[8:]
mab_ts = mab_ts[8:]
gom_ts = gom_ts[8:]

In [ ]:
fig2, ax = plt.subplots(nrows = 3, ncols = 1, figsize = (10,10))

ax[0].plot(chla.time,nyb_ts,'orange')
ax[0].set_ylim([0.5,2.5])
ax[0].tick_params(labelsize = 12)
ax[0].set_ylabel('Chla (mg $\mathregular{m^{-3}}$)', fontsize = 14, y=0.5)
ax2 = plt.twinx(ax[0])
ax2.set_ylabel('NYB', fontsize = 14)
ax2.set_yticks([])

ax[1].plot(chla.time,mab_ts,'purple')
ax[1].set_ylim([0.5,2.5])
ax[1].tick_params(labelsize = 12)
ax[1].set_ylabel('Chla (mg $\mathregular{m^{-3}}$)', fontsize = 14, y=0.5)
ax3 = plt.twinx(ax[1])
ax3.set_ylabel('MAB', fontsize = 14)
ax3.set_yticks([])

ax[2].plot(chla.time,gom_ts,'green')
ax[2].set_ylim([0.5,2.5])
ax[2].tick_params(labelsize = 12)
ax[2].set_ylabel('Chla (mg $\mathregular{m^{-3}}$)', fontsize = 14, y=0.5)
ax4 = plt.twinx(ax[2])
ax4.set_ylabel('GoM', fontsize = 14)
ax4.set_yticks([])

In [ ]:
YR = pn.DatetimeIndex(chla.time.values).year
#MO = pn.DatetimeIndex(chla.time.values).month

In [ ]:
MO = pn.to_datetime(chla.time.values, format='%m').month_name().str.slice(stop=3)
MO

In [ ]:
i=10
fig, ax = plt.subplots(figsize=(10,10), subplot_kw={'projection': ccrs.PlateCarree()})
extent = [285,290,38.5,41.5]
ax.set_extent(extent)
ax.gridlines()
ax.coastlines(resolution='50m')
C=plt.contourf(chla.lon,chla.lat,chlaNYB[4,:,:],levels = np.arange(0,9,0.5))
#C2 = ax.contour(X, Y, Z,levels=np.arange(0,25,2.5),colors = 'k',vmax = 80)
#ax.plot(360-X, Y, "ok", label="station")
#plt.plot(x, y, "ok", label="input point")
#plt.title('SON CTD 2019 MLD (temp)')
plt.colorbar(C)
#plt.clabel(C2)
ax.set_xticks(np.arange(-75,-69,1), crs=ccrs.PlateCarree())
ax.set_yticks(np.arange(38,42,0.5), crs=ccrs.PlateCarree())
ax.tick_params(labelsize=14)
#plt.contourf(chla.lon,chla.lat,chlaNYB[0,:,:])
plt.title('Satellite Chla, ' + MO[i] + ' ' + str(YR[i]), fontsize = 16, fontweight = 'bold')

In [ ]:
chla.lon


In [ ]:
#chlaNYB = chlaNYB.transpose([0,2,1])

In [ ]:
i=10
fig, ax = plt.subplots(figsize=(10,10), subplot_kw={'projection': ccrs.PlateCarree()})
extent = [285,290,38.5,41.5]
ax.set_extent(extent)
ax.gridlines()
ax.coastlines(resolution='50m')
C=plt.contourf(chla.lon,chla.lat[:125],chlaNYB[4,:125,:],levels = np.arange(0,9,0.5))
#C2 = ax.contour(X, Y, Z,levels=np.arange(0,25,2.5),colors = 'k',vmax = 80)
#ax.plot(360-X, Y, "ok", label="station")
#plt.plot(x, y, "ok", label="input point")
#plt.title('SON CTD 2019 MLD (temp)')
plt.colorbar(C)
#plt.clabel(C2)
ax.set_xticks(np.arange(-75,-69,1), crs=ccrs.PlateCarree())
ax.set_yticks(np.arange(38,42,0.5), crs=ccrs.PlateCarree())
ax.tick_params(labelsize=14)
#plt.contourf(chla.lon,chla.lat,chlaNYB[0,:,:])
plt.title('Satellite Chla, ' + MO[i] + ' ' + str(YR[i]), fontsize = 16, fontweight = 'bold')

In [ ]:
chla_NYB = xr.DataArray(data = chlaNYB, coords = {'time':chla.time, 'lon':chla.lon, 'lat':chla.lat}, dims = ['time','lon','lat'])

In [ ]:
#Chla_NYB = chla_NYB.transpose('time','lon','lat', transpose_coords = True)

In [ ]:
def custom_plotfunc(chlaNYB, fig, tt, framedim="time"):
    
    ax = fig.subplots(figsize=(10,10), subplot_kw={'projection': ccrs.PlateCarree()})
    extent = [285,290,38.5,41.5]
    ax.set_extent(extent)
    ax.gridlines()
    ax.coastlines(resolution='50m')
    C=plt.contourf(chlaNYB.lon,chlaNYB.lat,chlaNYB[tt,:,:],levels = np.arange(0,9,0.5))
    #C2 = ax.contour(X, Y, Z,levels=np.arange(0,25,2.5),colors = 'k',vmax = 80)
    #ax.plot(360-X, Y, "ok", label="station")
    #plt.plot(x, y, "ok", label="input point")
    #plt.title('SON CTD 2019 MLD (temp)')
    plt.colorbar(C)
    #plt.clabel(C2)
    ax.set_xticks(np.arange(-75,-69,1), crs=ccrs.PlateCarree())
    ax.set_yticks(np.arange(38,42,0.5), crs=ccrs.PlateCarree())
    ax.tick_params(labelsize=14)
    #plt.contourf(chla.lon,chla.lat,chlaNYB[0,:,:])
    plt.title('Satellite Chla, ' + MO[tt] + ' ' + str(YR[tt]), fontsize = 16, fontweight = 'bold')
    
    #fig.colorbar(scat, label='Depth [m]', shrink=0.8, ticks=np.arange(0, 6000, 1000))
    
    fig.subplots_adjust(top=0.8)
    
    return ax, C



In [ ]:
mov = Movie(chlaNYB)

In [ ]:
mov.preview(10)

In [ ]:
ds = hv.Dataset((chla.lon, chla.lat, chla.time, chlaNYB),
                ['lon', 'lat', 'time'], 'Chlorophyll a')
ds

In [ ]:
ds = ds.clone(datatype=['xarray']).data

In [ ]:
opts.defaults(
    opts.GridSpace(shared_xaxis=True, shared_yaxis=True),
    opts.Image(cmap='viridis', width=400, height=400),
    opts.Labels(text_color='white', text_font_size='8pt', text_align='left', text_baseline='bottom'),
    opts.Path(color='white'),
    opts.Spread(width=600),
    opts.Overlay(show_legend=False))


In [ ]:
ds.to(hv.Image, ['lon', 'lat']).hist()

In [ ]:
np.shape(chlaNYB)

In [ ]:
from holoviews.operation import contours

In [ ]:
img = hv.Image(chlaNYB[2,:,:])
filled_contours = contours(img, filled=True)

filled_contours

In [ ]:
chla_NYB

In [ ]:
holomap = hv.HoloMap([(t, hv.Image(chla_NYB[t,:,:])) for t in range(280)], kdims = 'time').opts(
    cmap='magma',  xaxis='bare', yaxis='bare')

contour_hmap = contours(holomap, filled=True).opts(cmap = 'viridis', colorbar=True, clim = (0,6),xlabel = '', clabel = 'chla')

#hv.save(contour_hmap, 'test_holoview.mp4', fmt='mp4', fps = 1)
hv.output(contour_hmap, holomap='mp4', fps=5)

In [ ]:
hv.save(contour_hmap, 'NYB_chla.mp4', fmt='mp4', fps = 5)

In [ ]:
hv.help(hv.Contours)


In [ ]:
ds = hv.Dataset((np.arange(50), np.arange(111), np.arange(62), calcium_array),
                ['Time', 'x', 'y'], 'Fluorescence')
ds

In [ ]:
monthly = np.empty([len(unique_years),12])

k=0
for i in (unique_years):
    year_picked = nyb_df[nyb_df.year == i]
    for j in range(12):
        if any(year_picked.month == float(j+1)):
            month_picked = year_picked[(year_picked.month == float(j+1))]
            if len(month_picked)<3: # Sets the minimum number of datapoints to 3 (you can pick)
                monthly_50[k,j] = np.nan
                monthly_bot[k,j] = np.nan
            else:
                monthly_50[k,j] = np.nanmean(month_picked.strat50)
                monthly_bot[k,j] = np.nanmean(month_picked.stratbot)
        else:
            monthly_50[k,j] = np.nan
            monthly_bot[k,j] = np.nan
    k = k+1

In [ ]:
%%opts Image [colorbar=True]
%%output holomap='mp4', fps=1
ds = hv.Dataset((np.arange(20), np.arange(20), np.arange(5), np.arange(2),
                    np.random.random((2, 5, 20, 20))), kdims=['w', 'x', 'y', 'z'], vdims=['A'])
ds.to(hv.Image).layout('z')